# Pre-race Predictions on a real race

Loads one race weekend through FastF1, makes every driver's Pre-race Prediction, compares it with the Strategies drivers actually ran, and prints the same table and race summary as the command:

```sh
uv run f1-strategy 2024 "Singapore Grand Prix"
```

Run from the repository root so the FastF1 cache in `.cache/fastf1` is reused.

In [1]:
import fastf1

from f1_strategy import NotDryRaceError, Settings, compare, predict
from f1_strategy.fastf1_loader import load_weekend
from f1_strategy.report import format_header, format_summary, format_table, strategy_label

fastf1.set_log_level("WARNING")

## Choose the race and settings

In [2]:
SEASON = 2024
EVENT = "Singapore Grand Prix"
settings = Settings()

In [3]:
weekend = load_weekend(SEASON, EVENT)
print(format_header(weekend.pre_race, settings))

core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 18)


2024 Marina Bay: 62 laps, Pit Loss 30.6 s
Settings: probability_scale_s=5.0, plausible_margin_s=20.0, stop_window_margin_s=1.0, fuel_effect_s_per_lap=0.03, min_long_run_length=4, long_run_slow_lap_margin_s=3.0, push_lap_fast_margin_s=1.0


## Predict and compare

In [4]:
predictions = predict(weekend, settings)
comparison = compare(predictions, weekend)
print(format_table(predictions, comparison))

Driver  #  Candidate Strategy       Prob  Stop Windows          Actual Strategy                     Result
----------------------------------------------------------------------------------------------------------
NOR     1  1-stop: S → H             32%  5–9                   1-stop: M → H (lap 30)              top-1 no · top-3 yes · Stop Window hits 0/1
        2  1-stop: H → S             32%  53–57
        3  1-stop: M → H             18%  3–6
VER     1  1-stop: M → H             50%  11–16                 1-stop: M → H (lap 29)              top-1 yes · top-3 yes · Stop Window hits 0/1
        2  1-stop: H → M             50%  46–51
PIA     1  1-stop: M → H             50%  3–6                   1-stop: M → H (lap 38)              top-1 yes · top-3 yes · Stop Window hits 0/1
        2  1-stop: H → M             50%  56–59
RUS     1  1-stop: S → M             17%  1–2                   1-stop: M → H (lap 27)              top-1 no · top-3 no · Stop Window hits 1/1
        2  1-stop: 

In [5]:
print(format_summary(comparison))

Scoreable drivers: 7 of 20
Top-1 Strategy Match: 2/7 (29%)
Top-3 Strategy Match: 6/7 (86%)
Stop Window hits: 1/7 actual stops (14%)
Neutralised Race: no


## Explore

One row per driver: the top Candidate Strategy and the actual Strategy, ready for filtering or charting.

In [6]:
import pandas as pd

rows = []
for driver, prediction in predictions.items():
    top = prediction.candidates[0]
    score = comparison.scores[driver]
    rows.append({
        "driver": driver,
        "top prediction": strategy_label(top.strategy.compounds),
        "probability": round(top.probability, 2),
        "Stop Windows": ", ".join(f"{s.first_lap}-{s.last_lap}" for s in top.stops),
        "actual Strategy": strategy_label(score.actual.compounds) if score.actual else None,
        "actual stop laps": score.actual.stop_laps if score.actual else None,
        "top-1": score.top_1_match,
        "top-3": score.top_3_match,
        "unscoreable": score.unscoreable,
    })
pd.DataFrame(rows).set_index("driver")

,top prediction,probability,Stop Windows,actual Strategy,actual stop laps,top-1,top-3,unscoreable
driver,,,,,,,,
NOR,1-stop: S → H,0.32,5-9,1-stop: M → H,"(30,)",False,True,False
VER,1-stop: M → H,0.50,11-16,1-stop: M → H,"(29,)",True,True,False
PIA,1-stop: M → H,0.50,3-6,1-stop: M → H,"(38,)",True,True,False
RUS,1-stop: S → M,0.17,1-2,1-stop: M → H,"(27,)",False,False,False
LEC,1-stop: S → H,0.25,1-2,1-stop: M → H,"(36,)",False,True,False
HAM,1-stop: S → M,0.17,1-2,1-stop: S → H,"(17,)",False,True,False
SAI,1-stop: S → H,0.25,1-2,1-stop: M → H,"(13,)",False,True,False
ALO,1-stop: M → H,0.39,16-21,None,None,False,False,True
HUL,1-stop: M → H,0.50,22-27,None,None,False,False,True


## Races outside scope are refused

Canada 2024 started on Intermediates, so it is not a Dry Race.

In [7]:
try:
    predict(load_weekend(2024, "Canadian Grand Prix"), settings)
except NotDryRaceError as refusal:
    print(refusal)

core        WARNING 	Fixed incorrect tyre stint information for driver '22'


2024 Montréal is not a Dry Race (race laps on Intermediate, Wet); Pre-race Predictions only cover Dry Races.
